In [1]:
# =========================================================
# IMPORTS
# =========================================================

import numpy as np
import pandas as pd

import mlflow
import mlflow.xgboost
import mlflow.lightgbm

import optuna

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mlflow.set_experiment(
    "lightgbm-xgboost-uber-demand-prediction"
)
# load the training and test data

train_data_path = "data/train_new.csv"
test_data_path = "data/test_new.csv"

train_df = pd.read_csv(train_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

test_df = pd.read_csv(test_data_path, parse_dates=["tpep_pickup_datetime"]).set_index("tpep_pickup_datetime")

In [3]:
import mlflow
import mlflow.lightgbm
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.model_selection import (
    TimeSeriesSplit
)

# ===================================================
# CREATE TIME FEATURES
# ===================================================

df = train_df.copy()

df.index = pd.to_datetime(
    df.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = (
    df.index.day_name()
)

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL FEATURES
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

df["rolling_mean_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(24)
    .mean()
)

df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(24)
    .std()
)

df["rolling_mean_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(168)
    .mean()
)

df["rolling_std_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(168)
    .std()
)

# ---------------------------------------------------
# EXTRA FEATURES
# ---------------------------------------------------

df["is_peak_hour"] = (
    df["hour"].isin(
        [7,8,9,17,18,19]
    )
).astype(int)

df["is_night"] = (
    df["hour"].isin(
        [0,1,2,3,4,5]
    )
).astype(int)

df["weekend_hour_interaction"] = (
    df["hour"] *
    df["is_weekend"]
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# TEST FEATURES
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# basic

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# cyclical

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# lag

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# rolling

test_processed["rolling_mean_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(24)
    .mean()
)

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(24)
    .std()
)

test_processed["rolling_mean_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(168)
    .mean()
)

test_processed["rolling_std_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(168)
    .std()
)

# extra

test_processed["is_peak_hour"] = (
    test_processed["hour"].isin(
        [7,8,9,17,18,19]
    )
).astype(int)

test_processed["is_night"] = (
    test_processed["hour"].isin(
        [0,1,2,3,4,5]
    )
).astype(int)

test_processed["weekend_hour_interaction"] = (
    test_processed["hour"] *
    test_processed["is_weekend"]
)

test_processed = (
    test_processed.dropna()
)

# ===================================================
# FEATURES / TARGET
# ===================================================

X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed[
    "total_pickups"
]

# ===================================================
# CATEGORY ENCODING
# ===================================================

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
        .cat.codes
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
        .cat.codes
    )

# ===================================================
# WAPE
# ===================================================

def wape(y_true, y_pred):

    return (
        np.sum(
            np.abs(y_true - y_pred)
        )
        /
        np.sum(
            np.abs(y_true)
        )
    ) * 100

# ===================================================
# TIME SERIES CV
# ===================================================

tscv = TimeSeriesSplit(
    n_splits=5
)

# ===================================================
# MLFLOW
# ===================================================

mlflow.set_experiment(
    "lightgbm_rmse_cv"
)

# ===================================================
# OBJECTIVE
# ===================================================

def objective(trial):

    rmse_scores = []

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            200,
            700,
            step=50
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            1e-2,
            0.1,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            8
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            20,
            150,
            step=10
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            10,
            50,
            step=5
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.7,
            1.0
        ),

        "colsample_bytree": (
            trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            )
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-4,
            5,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-4,
            5,
            log=True
        ),

        "objective": "regression",

        "random_state": 42,

        "n_jobs": -1
    }

    for train_idx, val_idx in tscv.split(X_full):

        X_train = X_full.iloc[
            train_idx
        ]

        X_val = X_full.iloc[
            val_idx
        ]

        y_train = y_full.iloc[
            train_idx
        ]

        y_val = y_full.iloc[
            val_idx
        ]

        model = LGBMRegressor(
            **params
        )

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_val
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        rmse_scores.append(
            rmse
        )

    return np.mean(
        rmse_scores
    )

# ===================================================
# OPTUNA
# ===================================================

study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=50
)

print(study.best_params)

# ===================================================
# FINAL MODEL
# ===================================================

final_model = LGBMRegressor(

    **study.best_params,

    objective="regression",

    random_state=42,

    n_jobs=-1
)

final_model.fit(
    X_full,
    y_full
)

# ===================================================
# TEST PREDICTIONS
# ===================================================

y_test_pred = final_model.predict(
    X_test
)

# ===================================================
# FINAL METRICS
# ===================================================

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_mape = mean_absolute_percentage_error(
    y_test,
    y_test_pred
)

final_wape = wape(
    y_test,
    y_test_pred
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

print("\nFINAL RESULTS")

print("MAE :", final_mae)

print("RMSE:", final_rmse)

print("MAPE:", final_mape)

print("WAPE:", final_wape)

print("R2  :", final_r2)

2026/05/28 05:05:44 INFO mlflow.tracking.fluent: Experiment with name 'lightgbm_rmse_cv' does not exist. Creating a new experiment.
[I 2026-05-28 05:05:44,304] A new study created in memory with name: no-name-f9f2678b-5546-42c2-a586-335c64aa9838


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002418 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2946
[LightGBM] [Info] Number of data points in the train set: 28752, number of used features: 22
[LightGBM] [Info] Start training from score 83.801649
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

[I 2026-05-28 05:06:24,986] Trial 0 finished with value: 28.20979714470807 and parameters: {'n_estimators': 600, 'learning_rate': 0.019281442471421138, 'max_depth': 6, 'num_leaves': 150, 'min_child_samples': 30, 'subsample': 0.9281350491654149, 'colsample_bytree': 0.796341985041068, 'reg_alpha': 0.11781479197578518, 'reg_lambda': 0.0022770165423298327}. Best is trial 0 with value: 28.20979714470807.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002288 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2946
[LightGBM] [Info] Number of data points in the train set: 28752, number of used features: 22
[LightGBM] [Info] Start training from score 83.801649
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

[I 2026-05-28 05:06:52,517] Trial 1 finished with value: 28.10335955576138 and parameters: {'n_estimators': 700, 'learning_rate': 0.02200609780320096, 'max_depth': 8, 'num_leaves': 20, 'min_child_samples': 40, 'subsample': 0.7373152450284567, 'colsample_bytree': 0.9845615382870281, 'reg_alpha': 0.00013944181402660979, 'reg_lambda': 0.11387360622852644}. Best is trial 1 with value: 28.10335955576138.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003103 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2946
[LightGBM] [Info] Number of data points in the train set: 28752, number of used features: 22
[LightGBM] [Info] Start training from score 83.801649
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

... (92 outputs left out)

[I 2026-05-28 05:16:51,558] Trial 48 finished with value: 28.43208460322409 and parameters: {'n_estimators': 450, 'learning_rate': 0.01042328598296881, 'max_depth': 8, 'num_leaves': 30, 'min_child_samples': 25, 'subsample': 0.7138273360025392, 'colsample_bytree': 0.9397252758037211, 'reg_alpha': 0.000102353073793478, 'reg_lambda': 0.0053376534229694255}. Best is trial 41 with value: 27.40499671045464.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001044 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2946
[LightGBM] [Info] Number of data points in the train set: 28752, number of used features: 22
[LightGBM] [Info] Start training from score 83.801649
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

[I 2026-05-28 05:17:02,242] Trial 49 finished with value: 28.877037970389914 and parameters: {'n_estimators': 600, 'learning_rate': 0.0399735209841235, 'max_depth': 4, 'num_leaves': 70, 'min_child_samples': 10, 'subsample': 0.73681148710146, 'colsample_bytree': 0.8184722280356105, 'reg_alpha': 0.00037187560871905224, 'reg_lambda': 0.02260361224001598}. Best is trial 41 with value: 27.40499671045464.


{'n_estimators': 350, 'learning_rate': 0.03299994815134275, 'max_depth': 3, 'num_leaves': 50, 'min_child_samples': 20, 'subsample': 0.7201581277315464, 'colsample_bytree': 0.9704138662053928, 'reg_alpha': 0.0004626059078153259, 'reg_lambda': 0.008116787345233904}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.025557 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2970
[LightGBM] [Info] Number of data points in the train set: 172512, number of used features: 22
[LightGBM] [Info] Start training from score 124.257362
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warn

In [ ]:
final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)
mlflow.log_metric(
    "FINAL_MAE",
    final_mae
)

In [6]:
print("\nFINAL RESULTS")

print("MAE :", final_mae)

print("RMSE:", final_rmse)

print("MAPE:", final_mape)

print("WAPE:", final_wape)

print("R2  :", final_r2)


FINAL RESULTS
MAE : 13.670388072025842
RMSE: 22.07524506870366
MAPE: 335036687010504.3
WAPE: 10.36125829670718
R2  : 0.9714776278320117


In [7]:
import mlflow
import mlflow.lightgbm
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.model_selection import (
    TimeSeriesSplit
)

# ===================================================
# CREATE TIME FEATURES
# ===================================================

df = train_df.copy()

df.index = pd.to_datetime(
    df.index
)

# ---------------------------------------------------
# BASIC TIME FEATURES
# ---------------------------------------------------

df["hour"] = df.index.hour

df["day"] = df.index.day

df["month"] = df.index.month

df["day_of_week"] = (
    df.index.day_name()
)

df["is_weekend"] = (
    df.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL FEATURES
# ---------------------------------------------------

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

df["lag_1"] = (
    df["total_pickups"]
    .shift(1)
)

df["lag_24"] = (
    df["total_pickups"]
    .shift(24)
)

df["lag_168"] = (
    df["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

df["rolling_mean_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(24)
    .mean()
)

df["rolling_std_24"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(24)
    .std()
)

df["rolling_mean_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(168)
    .mean()
)

df["rolling_std_168"] = (
    df["total_pickups"]
    .shift(1)
    .rolling(168)
    .std()
)

# ---------------------------------------------------
# EXTRA FEATURES
# ---------------------------------------------------

df["is_peak_hour"] = (
    df["hour"].isin(
        [7,8,9,17,18,19]
    )
).astype(int)

df["is_night"] = (
    df["hour"].isin(
        [0,1,2,3,4,5]
    )
).astype(int)

df["weekend_hour_interaction"] = (
    df["hour"] *
    df["is_weekend"]
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

df = df.dropna()

# ===================================================
# TEST FEATURES
# ===================================================

test_processed = test_df.copy()

test_processed.index = pd.to_datetime(
    test_processed.index
)

# ---------------------------------------------------
# BASIC FEATURES
# ---------------------------------------------------

test_processed["hour"] = (
    test_processed.index.hour
)

test_processed["day"] = (
    test_processed.index.day
)

test_processed["month"] = (
    test_processed.index.month
)

test_processed["day_of_week"] = (
    test_processed.index.day_name()
)

test_processed["is_weekend"] = (
    test_processed.index.dayofweek >= 5
).astype(int)

# ---------------------------------------------------
# CYCLICAL FEATURES
# ---------------------------------------------------

test_processed["hour_sin"] = np.sin(
    2 * np.pi * test_processed["hour"] / 24
)

test_processed["hour_cos"] = np.cos(
    2 * np.pi * test_processed["hour"] / 24
)

# ---------------------------------------------------
# LAG FEATURES
# ---------------------------------------------------

test_processed["lag_1"] = (
    test_processed["total_pickups"]
    .shift(1)
)

test_processed["lag_24"] = (
    test_processed["total_pickups"]
    .shift(24)
)

test_processed["lag_168"] = (
    test_processed["total_pickups"]
    .shift(168)
)

# ---------------------------------------------------
# ROLLING FEATURES
# ---------------------------------------------------

test_processed["rolling_mean_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(24)
    .mean()
)

test_processed["rolling_std_24"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(24)
    .std()
)

test_processed["rolling_mean_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(168)
    .mean()
)

test_processed["rolling_std_168"] = (
    test_processed["total_pickups"]
    .shift(1)
    .rolling(168)
    .std()
)

# ---------------------------------------------------
# EXTRA FEATURES
# ---------------------------------------------------

test_processed["is_peak_hour"] = (
    test_processed["hour"].isin(
        [7,8,9,17,18,19]
    )
).astype(int)

test_processed["is_night"] = (
    test_processed["hour"].isin(
        [0,1,2,3,4,5]
    )
).astype(int)

test_processed["weekend_hour_interaction"] = (
    test_processed["hour"] *
    test_processed["is_weekend"]
)

# ---------------------------------------------------
# DROP NULLS
# ---------------------------------------------------

test_processed = (
    test_processed.dropna()
)

# ===================================================
# FEATURES / TARGET
# ===================================================

X_full = df.drop(
    columns=["total_pickups"]
)

y_full = df["total_pickups"]

X_test = test_processed.drop(
    columns=["total_pickups"]
)

y_test = test_processed[
    "total_pickups"
]

# ===================================================
# CATEGORICAL FEATURES
# ===================================================

categorical_cols = [
    "region",
    "day_of_week"
]

for col in categorical_cols:

    X_full[col] = (
        X_full[col]
        .astype("category")
    )

    X_test[col] = (
        X_test[col]
        .astype("category")
    )

# ===================================================
# WAPE
# ===================================================

def wape(y_true, y_pred):

    return (
        np.sum(
            np.abs(y_true - y_pred)
        )
        /
        np.sum(
            np.abs(y_true)
        )
    ) * 100

# ===================================================
# TIME SERIES CV
# ===================================================

tscv = TimeSeriesSplit(
    n_splits=5
)

# ===================================================
# MLFLOW
# ===================================================

mlflow.set_experiment(
    "lightgbm_rmse_cv_final"
)

# ===================================================
# OBJECTIVE
# ===================================================

def objective(trial):

    rmse_scores = []

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            200,
            1000,
            step=50
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            1e-2,
            0.1,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            7
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            20,
            80,
            step=5
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            10,
            40,
            step=5
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.7,
            1.0
        ),

        "colsample_bytree": (
            trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            )
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-4,
            5,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-4,
            5,
            log=True
        ),

        "objective": "regression",

        "random_state": 42,

        "n_jobs": -1,

        "verbosity": -1
    }

    # ===============================================
    # CROSS VALIDATION
    # ===============================================

    for train_idx, val_idx in tscv.split(X_full):

        X_train = X_full.iloc[
            train_idx
        ]

        X_val = X_full.iloc[
            val_idx
        ]

        y_train = y_full.iloc[
            train_idx
        ]

        y_val = y_full.iloc[
            val_idx
        ]

        model = LGBMRegressor(
            **params
        )

        model.fit(

            X_train,
            y_train,

            eval_set=[
                (X_val, y_val)
            ],

            eval_metric="rmse",

            categorical_feature=
            categorical_cols,

            callbacks=[

                lgb.early_stopping(
                    50
                ),

                lgb.log_evaluation(0)
            ]
        )

        y_pred = model.predict(
            X_val
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val,
                y_pred
            )
        )

        rmse_scores.append(
            rmse
        )

    return np.mean(
        rmse_scores
    )

# ===================================================
# OPTUNA SEARCH
# ===================================================

study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=50
)

print("\nBest Params:")

print(study.best_params)

print("\nBest CV RMSE:")

print(study.best_value)

# ===================================================
# FINAL MODEL
# ===================================================

final_model = LGBMRegressor(

    **study.best_params,

    objective="regression",

    random_state=42,

    n_jobs=-1,

    verbosity=-1
)

# ===================================================
# TRAIN FINAL MODEL
# ===================================================

final_model.fit(

    X_full,
    y_full,

    categorical_feature=
    categorical_cols
)

# ===================================================
# TEST PREDICTIONS
# ===================================================

y_test_pred = final_model.predict(
    X_test
)

# ===================================================
# FINAL METRICS
# ===================================================

final_mae = mean_absolute_error(
    y_test,
    y_test_pred
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

final_wape = wape(
    y_test,
    y_test_pred
)

final_r2 = r2_score(
    y_test,
    y_test_pred
)

# ===================================================
# PRINT RESULTS
# ===================================================

print("\nFINAL RESULTS")

print("MAE :", final_mae)

print("RMSE:", final_rmse)

print("WAPE:", final_wape)

print("R2  :", final_r2)

# ===================================================
# LOG FINAL MODEL
# ===================================================

with mlflow.start_run(

    run_name=
    "best_lightgbm_model"

):

    mlflow.log_params(
        study.best_params
    )

    mlflow.log_metric(
        "FINAL_MAE",
        final_mae
    )

    mlflow.log_metric(
        "FINAL_RMSE",
        final_rmse
    )

    mlflow.log_metric(
        "FINAL_WAPE",
        final_wape
    )

    mlflow.log_metric(
        "FINAL_R2",
        final_r2
    )

    mlflow.lightgbm.log_model(
        final_model,
        artifact_path="model"
    )

print(
    "\nFinal LightGBM model logged successfully."
)

2026/05/28 05:44:30 INFO mlflow.tracking.fluent: Experiment with name 'lightgbm_rmse_cv_final' does not exist. Creating a new experiment.
[I 2026-05-28 05:44:30,917] A new study created in memory with name: no-name-fca54f41-a947-4a7a-9c9b-d664f0d8a73f


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[267]	valid_0's rmse: 35.8033	valid_0's l2: 1281.88
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[236]	valid_0's rmse: 46.1005	valid_0's l2: 2125.25
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[282]	valid_0's rmse: 24.391	valid_0's l2: 594.919
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[340]	valid_0's rmse: 20.2917	valid_0's l2: 411.755
Training until validation scores don't improve for 50 rounds


[I 2026-05-28 05:44:54,470] Trial 0 finished with value: 28.169035435058674 and parameters: {'n_estimators': 750, 'learning_rate': 0.02698054308191676, 'max_depth': 6, 'num_leaves': 25, 'min_child_samples': 10, 'subsample': 0.7887921120074913, 'colsample_bytree': 0.9107565785927755, 'reg_alpha': 0.004234145619237375, 'reg_lambda': 0.002770935710556404}. Best is trial 0 with value: 28.169035435058674.


Did not meet early stopping. Best iteration is:
[750]	valid_0's rmse: 14.2587	valid_0's l2: 203.31
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[450]	valid_0's rmse: 36.4987	valid_0's l2: 1332.16
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[449]	valid_0's rmse: 45.9424	valid_0's l2: 2110.7
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[450]	valid_0's rmse: 24.6765	valid_0's l2: 608.93
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[450]	valid_0's rmse: 20.7111	valid_0's l2: 428.948
Training until validation scores don't improve for 50 rounds


[I 2026-05-28 05:45:12,503] Trial 1 finished with value: 28.477510283226593 and parameters: {'n_estimators': 450, 'learning_rate': 0.011646256586713682, 'max_depth': 5, 'num_leaves': 45, 'min_child_samples': 15, 'subsample': 0.7034581119867421, 'colsample_bytree': 0.8386509022421719, 'reg_alpha': 0.0003755246684773071, 'reg_lambda': 0.15573128825779717}. Best is trial 0 with value: 28.169035435058674.


Did not meet early stopping. Best iteration is:
[450]	valid_0's rmse: 14.5589	valid_0's l2: 211.961
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[472]	valid_0's rmse: 35.3051	valid_0's l2: 1246.45
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[455]	valid_0's rmse: 45.694	valid_0's l2: 2087.94
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[548]	valid_0's rmse: 23.8971	valid_0's l2: 571.073
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[457]	valid_0's rmse: 20.5363	valid_0's l2: 421.741
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[644]	valid_0's rmse: 14.465	valid_0's l2: 209.238


... (93 outputs left out)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[269]	valid_0's rmse: 34.8341	valid_0's l2: 1213.42
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[391]	valid_0's rmse: 45.3064	valid_0's l2: 2052.67
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[495]	valid_0's rmse: 23.9614	valid_0's l2: 574.15
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[262]	valid_0's rmse: 20.4806	valid_0's l2: 419.453
Training until validation scores don't improve for 50 rounds


[I 2026-05-28 06:02:09,027] Trial 49 finished with value: 27.76145907480774 and parameters: {'n_estimators': 700, 'learning_rate': 0.0280128894144738, 'max_depth': 4, 'num_leaves': 50, 'min_child_samples': 40, 'subsample': 0.8670857461032204, 'colsample_bytree': 0.7597005830358013, 'reg_alpha': 1.522253982842798, 'reg_lambda': 1.0621441351524694}. Best is trial 17 with value: 27.751544782782503.


Did not meet early stopping. Best iteration is:
[686]	valid_0's rmse: 14.2248	valid_0's l2: 202.345

Best Params:
{'n_estimators': 900, 'learning_rate': 0.041141446696751904, 'max_depth': 6, 'num_leaves': 40, 'min_child_samples': 40, 'subsample': 0.8944927218347417, 'colsample_bytree': 0.7385395093856881, 'reg_alpha': 0.015473270687785787, 'reg_lambda': 0.003873307086930742}

Best CV RMSE:
27.751544782782503

FINAL RESULTS
MAE : 12.134407903131995
RMSE: 19.791609343941666
WAPE: 9.197085986113027
R2  : 0.977073550907221


Exception: Run with UUID 9d062824b3cc419f867cb4d469b2e41e is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

In [5]:
"""
NYC Taxi Demand Forecasting — Full Feature Set, Zero Row Dropping
================================================================
Key change from previous version: dropna() is replaced with
targeted imputation so ALL rows are kept, including early rows
where long lags (lag_168, lag_336, lag_720) would have been NaN.

Imputation strategy per feature type:
  rolling windows  → min_periods=1  (partial window from row 1)
  lag NaNs         → expanding mean fallback + _available flag
  region lags      → same, but expanding mean is per-region
  weather join     → ffill(limit=2) + median fallback

The _available flags (lag_168_available, lag_336_available,
lag_720_available) tell LightGBM which lag values are real vs
imputed. The model learns to trust them when present and
discount when absent — better than just dropping those rows.
"""

import requests
import mlflow
import mlflow.lightgbm
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit


# ===================================================
# HELPER: impute a lag series with expanding mean
# ===================================================

def impute_lag(series: pd.Series) -> pd.Series:
    """
    Fill NaNs in a lag/shift series with the expanding mean
    computed from available (non-NaN) values up to that point.
    Expanding mean is the best unbiased estimate when we have
    no lookback data yet.
    """
    expanding_mean = series.expanding(min_periods=1).mean()
    return series.fillna(expanding_mean)


def impute_lag_grouped(df: pd.DataFrame, col: str, group_col: str) -> pd.Series:
    """
    Same imputation but applied per group (e.g. per region).
    """
    def _impute(grp):
        exp = grp.expanding(min_periods=1).mean()
        return grp.fillna(exp)
    return df.groupby(group_col)[col].transform(_impute)


# ===================================================
# WEATHER DATA — Open-Meteo Historical API
# ===================================================

def fetch_nyc_weather_2016(
    start_date: str = "2016-01-01",
    end_date:   str = "2016-03-31",
) -> pd.DataFrame:
    """
    Hourly NYC weather from Open-Meteo free historical archive.
    No API key required.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   40.7128,
        "longitude": -74.0060,
        "start_date": start_date,
        "end_date":   end_date,
        "hourly": ",".join([
            "temperature_2m",
            "apparent_temperature",
            "precipitation",
            "snowfall",
            "snow_depth",
            "windspeed_10m",
            "weathercode",
            "cloudcover",
            "relativehumidity_2m",
        ]),
        "timezone": "America/New_York",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    weather = pd.DataFrame(data["hourly"])
    weather["time"] = pd.to_datetime(weather["time"])
    weather = weather.set_index("time")

    def wmo_to_condition(code):
        if pd.isna(code):    return "unknown"
        code = int(code)
        if code in [0, 1]:           return "clear"
        elif code in [2, 3]:         return "cloudy"
        elif code in range(51, 68):  return "rain"
        elif code in range(71, 78):  return "snow"
        elif code in range(80, 83):  return "rain"
        elif code in range(85, 87):  return "snow"
        elif code in [95, 96, 99]:   return "storm"
        else:                        return "other"

    weather["weather_condition"] = weather["weathercode"].apply(wmo_to_condition)
    weather["is_raining"]        = (weather["precipitation"] > 0).astype(int)
    weather["is_snowing"]        = (weather["snowfall"] > 0).astype(int)
    weather["is_heavy_weather"]  = (
        (weather["precipitation"] > 5) | (weather["snowfall"] > 2)
    ).astype(int)
    weather["is_freezing"]       = (weather["temperature_2m"] < 0).astype(int)
    weather["temp_bin"]          = pd.cut(
        weather["temperature_2m"],
        bins=[-40, 0, 10, 40],
        labels=["cold", "mild", "warm"],
    )
    # Winter Storm Jonas — Jan 22-25, NYC record snowfall ~26.8 in
    weather["is_storm_jonas"] = (
        (weather.index >= "2016-01-22") &
        (weather.index <= "2016-01-25")
    ).astype(int)

    return weather


# ===================================================
# HOLIDAYS & EVENTS — NYC Jan–Mar 2016
# ===================================================

NYC_HOLIDAYS_2016 = {
    pd.Timestamp("2016-01-01"): "new_years_day",
    pd.Timestamp("2016-01-18"): "mlk_day",
    pd.Timestamp("2016-02-15"): "presidents_day",
    pd.Timestamp("2016-02-14"): "valentines_day",
    pd.Timestamp("2016-03-17"): "st_patricks_day",
    pd.Timestamp("2016-03-25"): "good_friday",
    pd.Timestamp("2016-03-27"): "easter_sunday",
}

NYC_EVENTS_2016 = {
    "restaurant_week":   ("2016-01-18", "2016-01-29"),
    "fashion_week":      ("2016-02-11", "2016-02-18"),
    "nyc_half_marathon": ("2016-03-20", "2016-03-20"),
}


def build_holiday_features(df: pd.DataFrame) -> pd.DataFrame:
    date_only = df.index.normalize()
    df["is_holiday"]   = date_only.isin(NYC_HOLIDAYS_2016.keys()).astype(int)
    df["holiday_name"] = date_only.map(NYC_HOLIDAYS_2016).fillna("none")

    holiday_dates = sorted(NYC_HOLIDAYS_2016.keys())

    def days_to_next(ts):
        future = [h for h in holiday_dates if h >= ts.normalize()]
        return (future[0] - ts.normalize()).days if future else 99

    def days_after_prev(ts):
        past = [h for h in holiday_dates if h <= ts.normalize()]
        return (ts.normalize() - past[-1]).days if past else 99

    df["days_to_holiday"]    = [days_to_next(t)    for t in df.index]
    df["days_after_holiday"] = [days_after_prev(t) for t in df.index]

    for event_name, (start, end) in NYC_EVENTS_2016.items():
        df[f"is_{event_name}"] = (
            (df.index >= pd.Timestamp(start)) &
            (df.index <= pd.Timestamp(end) + pd.Timedelta(hours=23))
        ).astype(int)

    return df


# ===================================================
# CORE FEATURE BUILDER — no dropna()
# ===================================================

def build_features(
    raw_df:     pd.DataFrame,
    weather_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Full feature engineering pipeline.
    Zero rows are dropped — all NaNs are imputed.

    Parameters
    ----------
    raw_df     : pickup DataFrame, datetime index, 'total_pickups' column
    weather_df : output of fetch_nyc_weather_2016()
    """
    df = raw_df.copy()
    df.index = pd.to_datetime(df.index)

    # ------------------------------------------------
    # 1. BASIC TIME FEATURES
    # ------------------------------------------------
    df["hour"]        = df.index.hour
    df["day"]         = df.index.day
    df["month"]       = df.index.month
    df["day_of_week"] = df.index.dayofweek     # 0=Mon … 6=Sun
    df["is_weekend"]  = (df.index.dayofweek >= 5).astype(int)

    # ------------------------------------------------
    # 2. CYCLICAL ENCODING — hour, dow, day, month
    # ------------------------------------------------
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"]        / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"]        / 24)
    df["dow_sin"]   = np.sin(2 * np.pi * df["day_of_week"] /  7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["day_of_week"] /  7)
    df["day_sin"]   = np.sin(2 * np.pi * df["day"]         / 31)
    df["day_cos"]   = np.cos(2 * np.pi * df["day"]         / 31)
    df["month_sin"] = np.sin(2 * np.pi * df["month"]       / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"]       / 12)

    # ------------------------------------------------
    # 3. PEAK / NIGHT FLAGS
    # ------------------------------------------------
    df["is_peak_hour"] = df["hour"].isin([7,8,9,17,18,19]).astype(int)
    df["is_night"]     = df["hour"].isin([0,1,2,3,4,5]).astype(int)

    # ================================================
    # 4. GLOBAL LAG FEATURES (ALL KEPT)
    #    NaNs filled with expanding mean.
    #    Availability flag added for lags >= 24 h.
    # ================================================

    lag_shifts = {
        "lag_1":   1,
        "lag_24":  24,
        "lag_168": 168,
        "lag_336": 336,
        "lag_720": 720,
    }

    for col, shift in lag_shifts.items():
        raw_lag         = df["total_pickups"].shift(shift)
        available_flag  = raw_lag.notna().astype(int)
        df[col]         = impute_lag(raw_lag)
        if shift >= 24:
            df[f"{col}_available"] = available_flag

    # ------------------------------------------------
    # 5. ROLLING FEATURES — min_periods=1, no NaN
    # ------------------------------------------------
    base = df["total_pickups"].shift(1)          # prevent leakage

    df["rolling_mean_24"]  = base.rolling(24,  min_periods=1).mean()
    df["rolling_std_24"]   = base.rolling(24,  min_periods=1).std().fillna(0)
    df["rolling_mean_168"] = base.rolling(168, min_periods=1).mean()
    df["rolling_std_168"]  = base.rolling(168, min_periods=1).std().fillna(0)
    df["rolling_mean_336"] = base.rolling(336, min_periods=1).mean()
    df["rolling_mean_720"] = base.rolling(720, min_periods=1).mean()

    # ------------------------------------------------
    # 6. WEEKEND / WEEKDAY CONDITIONED ROLLING
    # ------------------------------------------------
    weekend_mask  = (df.index.dayofweek >= 5)
    weekday_mask  = ~weekend_mask

    weekend_base  = base.where(weekend_mask)
    weekday_base  = base.where(weekday_mask)

    df["rolling_mean_weekend_24"] = (
        weekend_base
        .rolling(24, min_periods=1)
        .mean()
        .fillna(df["rolling_mean_24"])   # fall back to global mean
    )
    df["rolling_mean_weekday_24"] = (
        weekday_base
        .rolling(24, min_periods=1)
        .mean()
        .fillna(df["rolling_mean_24"])
    )

    # ------------------------------------------------
    # 7. HOLIDAY FEATURES
    # ------------------------------------------------
    df = build_holiday_features(df)

    holiday_base = base.where(df["is_holiday"].astype(bool))
    df["rolling_mean_holiday_24"] = (
        holiday_base
        .rolling(24, min_periods=1)
        .mean()
        .fillna(df["rolling_mean_24"])
    )

    # ================================================
    # 8. REGION LAG & ROLLING — safe sort pattern
    #
    #   Correct approach:
    #     1. sort_index() so time is monotone within each region
    #     2. groupby(sort=False) — preserves time-sorted order
    #     3. impute NaNs within each region using expanding mean
    #     4. sort_index() again to restore global time order
    # ================================================

    if "region" in df.columns:

        df = df.sort_index()      # ensure time order before groupby

        grp_col = "region"

        # Region lags
        for col, shift in [("region_lag_1", 1), ("region_lag_24", 24),
                            ("region_lag_168", 168), ("region_lag_336", 336)]:
            raw_lag = df.groupby(grp_col, sort=False)["total_pickups"].shift(shift)
            df[col] = impute_lag_grouped(
                df.assign(**{col: raw_lag}), col, grp_col
            )
            if shift >= 24:
                df[f"{col}_available"] = raw_lag.notna().astype(int)

        # Region rolling — min_periods=1, no NaN
        shifted_grp = (
            df.groupby(grp_col, sort=False)["total_pickups"]
            .shift(1)
        )
        df["_base_r"] = shifted_grp

        df["region_rolling_mean_24"] = (
            df.groupby(grp_col, sort=False)["_base_r"]
            .transform(lambda x: x.rolling(24, min_periods=1).mean())
        )
        df["region_rolling_std_24"] = (
            df.groupby(grp_col, sort=False)["_base_r"]
            .transform(lambda x: x.rolling(24, min_periods=1).std().fillna(0))
        )
        df["region_rolling_mean_168"] = (
            df.groupby(grp_col, sort=False)["_base_r"]
            .transform(lambda x: x.rolling(168, min_periods=1).mean())
        )

        df = df.drop(columns=["_base_r"])
        df = df.sort_index()      # restore global time order

    # ================================================
    # 9. WEATHER FEATURES
    # ================================================
    weather_cols = [
        "temperature_2m", "apparent_temperature",
        "precipitation", "snowfall", "snow_depth",
        "windspeed_10m", "cloudcover", "relativehumidity_2m",
        "is_raining", "is_snowing", "is_heavy_weather",
        "is_freezing", "is_storm_jonas",
        "temp_bin", "weather_condition",
    ]
    df = df.join(weather_df[weather_cols], how="left")

    # Short gaps: forward fill up to 2 hours
    df[weather_cols] = df[weather_cols].ffill(limit=2)

    # Remaining NaN → median per column (robust to outliers)
    for col in weather_cols:
        if df[col].isna().any():
            if df[col].dtype.kind in "ifc":           # numeric
                df[col] = df[col].fillna(df[col].median())
            else:                                      # categorical
                df[col] = df[col].fillna(df[col].mode().iloc[0])

    # ------------------------------------------------
    # 10. INTERACTION FEATURES
    # ------------------------------------------------
    df["weekend_hour_interaction"] = df["hour"]          * df["is_weekend"]
    df["rain_x_peak_hour"]         = df["is_raining"]    * df["is_peak_hour"]
    df["snow_x_night"]             = df["is_snowing"]    * df["is_night"]
    df["holiday_x_hour"]           = df["is_holiday"]    * df["hour"]
    df["temp_x_weekend"]           = df["temperature_2m"] * df["is_weekend"]
    df["storm_x_peak"]             = df["is_storm_jonas"] * df["is_peak_hour"]
    df["wind_x_rain"]              = df["windspeed_10m"]  * df["is_raining"]

    # ------------------------------------------------
    # 11. SANITY CHECK — no NaN should remain
    # ------------------------------------------------
    nan_counts = df.isnull().sum()
    remaining  = nan_counts[nan_counts > 0]
    if not remaining.empty:
        print("[WARNING] NaN still present after imputation:")
        print(remaining)
    else:
        print(f"[OK] Zero NaN rows. Shape: {df.shape}")

    return df


# ===================================================
# FETCH WEATHER
# ===================================================

print("Fetching NYC weather Jan–Mar 2016 ...")
weather_df = fetch_nyc_weather_2016()
print(f"  Weather shape: {weather_df.shape}")

# ===================================================
# BUILD TRAIN / TEST FEATURES
# ===================================================

print("Building train features ...")
df             = build_features(train_df, weather_df)

print("Building test features ...")
test_processed = build_features(test_df, weather_df)

# ===================================================
# FEATURES / TARGET
# ===================================================

X_full = df.drop(columns=["total_pickups"])
y_full = df["total_pickups"]

X_test = test_processed.drop(columns=["total_pickups"])
y_test = test_processed["total_pickups"]

# ===================================================
# CATEGORICAL FEATURES
# ===================================================

categorical_cols = [
    "region",
    "holiday_name",
    "weather_condition",
    "temp_bin",
]

for col in categorical_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].astype("category")
    if col in X_test.columns:
        X_test[col] = X_test[col].astype("category")

# ===================================================
# WAPE METRIC
# ===================================================

def wape(y_true, y_pred):
    return (
        np.sum(np.abs(y_true - y_pred)) /
        np.sum(np.abs(y_true))
    ) * 100

# ===================================================
# TIME-SERIES CROSS-VALIDATION
# ===================================================

tscv = TimeSeriesSplit(n_splits=5)

print(f"\nDataset has {len(X_full)} rows for CV.")
print(f"Min fold size at n_splits=5: ~{len(X_full) // 6} rows")

# ===================================================
# MLFLOW EXPERIMENT
# ===================================================

mlflow.set_experiment("lightgbm_full_features_no_drop")

# ===================================================
# OPTUNA OBJECTIVE
# ===================================================

def objective(trial):
    rmse_scores = []

    params = {
        "n_estimators":      trial.suggest_int("n_estimators",       200, 1000, step=50),
        "learning_rate":     trial.suggest_float("learning_rate",    1e-2, 0.1, log=True),
        "max_depth":         trial.suggest_int("max_depth",          3, 7),
        "num_leaves":        trial.suggest_int("num_leaves",         20, 80, step=5),
        "min_child_samples": trial.suggest_int("min_child_samples",  10, 40, step=5),
        "subsample":         trial.suggest_float("subsample",        0.7, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha",        1e-4, 5, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda",       1e-4, 5, log=True),
        "objective":         "regression",
        "random_state":      42,
        "n_jobs":            -1,
        "verbosity":         -1,
    }

    active_cat = [c for c in categorical_cols if c in X_full.columns]

    for train_idx, val_idx in tscv.split(X_full):
        X_tr, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            categorical_feature=active_cat,
            callbacks=[
                lgb.early_stopping(50),
                lgb.log_evaluation(0),
            ],
        )

        y_pred = model.predict(X_val)
        rmse_scores.append(
            np.sqrt(mean_squared_error(y_val, y_pred))
        )

    return np.mean(rmse_scores)

# ===================================================
# OPTUNA STUDY
# ===================================================

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("\nBest params:")
print(study.best_params)
print(f"\nBest CV RMSE: {study.best_value:.4f}")

# ===================================================
# FINAL MODEL
# ===================================================

active_cat = [c for c in categorical_cols if c in X_full.columns]

final_model = LGBMRegressor(
    **study.best_params,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
final_model.fit(X_full, y_full, categorical_feature=active_cat)

# ===================================================
# EVALUATION
# ===================================================

y_test_pred = final_model.predict(X_test)

final_mae  = mean_absolute_error(y_test, y_test_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
final_wape = wape(y_test, y_test_pred)
final_r2   = r2_score(y_test, y_test_pred)

print("\n--- FINAL TEST RESULTS ---")
print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"WAPE : {final_wape:.4f}%")
print(f"R2   : {final_r2:.4f}")

# ===================================================
# FEATURE IMPORTANCES
# ===================================================

feat_imp = pd.Series(
    final_model.feature_importances_,
    index=X_full.columns,
).sort_values(ascending=False)

print("\n--- TOP 20 FEATURE IMPORTANCES ---")
print(feat_imp.head(20).to_string())

# ===================================================
# MLFLOW LOGGING
# ===================================================

with mlflow.start_run(run_name="lgbm_full_features_no_drop"):

    mlflow.log_params(study.best_params)
    mlflow.log_metric("FINAL_MAE",  final_mae)
    mlflow.log_metric("FINAL_RMSE", final_rmse)
    mlflow.log_metric("FINAL_WAPE", final_wape)
    mlflow.log_metric("FINAL_R2",   final_r2)
    mlflow.log_param("n_features",  X_full.shape[1])
    mlflow.log_param("n_train_rows", len(X_full))

    mlflow.lightgbm.log_model(final_model, artifact_path="model")

print("\nModel logged to MLflow successfully.")

Fetching NYC weather Jan–Mar 2016 ...
  Weather shape: (2184, 16)
Building train features ...
[WARNING] NaN still present after imputation:
lag_1                          1
lag_24                        24
lag_168                      168
lag_336                      336
lag_720                      720
rolling_mean_24                1
rolling_mean_168               1
rolling_mean_336               1
rolling_mean_720               1
rolling_mean_weekend_24        1
rolling_mean_weekday_24        1
rolling_mean_holiday_24        1
region_lag_1                  30
region_lag_24                720
region_lag_168              5040
region_lag_336             10080
region_rolling_mean_24        30
region_rolling_mean_168       30
dtype: int64
Building test features ...
[WARNING] NaN still present after imputation:
lag_1                          1
lag_24                        24
lag_168                      168
lag_336                      336
lag_720                      720
rolling_mean_24

[I 2026-05-31 22:33:29,239] A new study created in memory with name: no-name-04de17e4-44af-4678-9305-2f8fa94f53e5



Dataset has 172680 rows for CV.
Min fold size at n_splits=5: ~28780 rows
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 21.1294	valid_0's l2: 446.452
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 18.3225	valid_0's l2: 335.715
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 21.7157	valid_0's l2: 471.574
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 20.7895	valid_0's l2: 432.204
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's rmse: 21.873	valid_0's l2: 478.427


[I 2026-05-31 22:36:25,334] Trial 0 finished with value: 20.76604006818473 and parameters: {'n_estimators': 1000, 'learning_rate': 0.014351933212897088, 'max_depth': 3, 'num_leaves': 70, 'min_child_samples': 30, 'subsample': 0.9482979676662767, 'colsample_bytree': 0.7416540653008956, 'reg_alpha': 0.00016229859658490006, 'reg_lambda': 0.03391763172965806}. Best is trial 0 with value: 20.76604006818473.


Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[600]	valid_0's rmse: 19.9398	valid_0's l2: 397.595
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[523]	valid_0's rmse: 17.4219	valid_0's l2: 303.522
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[600]	valid_0's rmse: 20.2898	valid_0's l2: 411.677
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[600]	valid_0's rmse: 19.583	valid_0's l2: 383.495
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[599]	valid_0's rmse: 20.1345	valid_0's l2: 405.399


[I 2026-05-31 22:39:18,230] Trial 1 finished with value: 19.473805252418238 and parameters: {'n_estimators': 600, 'learning_rate': 0.032982889616775084, 'max_depth': 5, 'num_leaves': 20, 'min_child_samples': 10, 'subsample': 0.8624359236010154, 'colsample_bytree': 0.7484620435971948, 'reg_alpha': 0.0006756088859977343, 'reg_lambda': 0.04971563532611577}. Best is trial 1 with value: 19.473805252418238.


... (95 outputs left out)

[I 2026-05-31 23:22:55,406] Trial 49 finished with value: 18.928585000564674 and parameters: {'n_estimators': 650, 'learning_rate': 0.05859258806100104, 'max_depth': 7, 'num_leaves': 55, 'min_child_samples': 20, 'subsample': 0.8940560704737123, 'colsample_bytree': 0.822532091858304, 'reg_alpha': 0.041165114959933656, 'reg_lambda': 0.3518171233608463}. Best is trial 29 with value: 18.91465159479538.



Best params:
{'n_estimators': 800, 'learning_rate': 0.05598964242281479, 'max_depth': 7, 'num_leaves': 60, 'min_child_samples': 30, 'subsample': 0.9352532803763558, 'colsample_bytree': 0.8984678707218833, 'reg_alpha': 0.00033289139268951327, 'reg_lambda': 0.03857858157470467}

Best CV RMSE: 18.9147

--- FINAL TEST RESULTS ---
MAE  : 12.4594
RMSE : 20.1163
WAPE : 9.4542%
R2   : 0.9763

--- TOP 20 FEATURE IMPORTANCES ---
region                      3669
lag_1                       3445
lag_2                       2411
lag_4                       2255
lag_3                       2013
hour                        1606
lag_24                      1309
avg_pickups                 1269
lag_168                     1256
hour_sin                    1237
lag_720                     1218
hour_cos                    1199
region_lag_1                1058
rolling_std_24              1052
region_lag_336              1033
lag_336                      962
rolling_std_168              949
region_lag_168 

2026/05/31 23:23:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/31 23:23:08 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Model logged to MLflow successfully.


In [11]:
# total NaN count in each column

df.isna().sum().sum()

np.int64(17186)

In [12]:
nan_report = pd.DataFrame({

    "nan_count": df.isna().sum(),

    "nan_percent": (
        df.isna().mean() * 100
    )

})

nan_report = nan_report[
    nan_report["nan_count"] > 0
]

print(nan_report)

                         nan_count  nan_percent
lag_1                            1     0.000579
lag_24                          24     0.013899
lag_168                        168     0.097290
lag_336                        336     0.194580
lag_720                        720     0.416956
rolling_mean_24                  1     0.000579
rolling_mean_168                 1     0.000579
rolling_mean_336                 1     0.000579
rolling_mean_720                 1     0.000579
rolling_mean_weekend_24          1     0.000579
rolling_mean_weekday_24          1     0.000579
rolling_mean_holiday_24          1     0.000579
region_lag_1                    30     0.017373
region_lag_24                  720     0.416956
region_lag_168                5040     2.918694
region_lag_336               10080     5.837387
region_rolling_mean_24          30     0.017373
region_rolling_mean_168         30     0.017373


In [ ]:
print("TRAIN NaNs")
print(train_df.isna().sum())

print("\nTEST NaNs")
print(test_df.isna().sum())

TRAIN NaNs
lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64

TEST NaNs
lag_1            0
lag_2            0
lag_3            0
lag_4            0
region           0
total_pickups    0
avg_pickups      0
day_of_week      0
dtype: int64


In [16]:
"""
NYC Taxi Demand Forecasting — Enterprise-Grade Pipeline v2
==========================================================
Upgrades applied on top of the zero-row-dropping baseline:

  [1] TimeSeries CV Gap      – tscv gap=8 (2-hour pipeline latency)
  [2] Optuna Pruning         – LightGBMPruningCallback stops bad trials early
  [3] Smart Imputation       – region × dow × hour mean before expanding mean
  [4] Memory Optimisation    – reduce_mem_usage() downcasts float64→float32,
                                int64→smallest safe int
  [5] API Robustness         – fetch_nyc_weather() derives dates from df.index;
                                tenacity retries on network / rate-limit errors

Zero-row-dropping strategy (expanding-mean fallback) is fully preserved.
"""

# ───────────────────────────────────────────────────────────────────────────
# IMPORTS
# ───────────────────────────────────────────────────────────────────────────

import requests
import mlflow
import mlflow.lightgbm
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

# [2] Optuna pruning integration
from optuna.integration import LightGBMPruningCallback

# [5] Tenacity retry decorators
from tenacity import retry, wait_exponential, stop_after_attempt, RetryError


# ───────────────────────────────────────────────────────────────────────────
# [4] MEMORY OPTIMISATION — reduce_mem_usage()
# ───────────────────────────────────────────────────────────────────────────

def reduce_mem_usage(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Downcast numeric columns to the smallest safe dtype to reduce RAM
    before passing the DataFrame to LightGBM.

    Strategy
    --------
    - float64  →  float32   (LightGBM never needs float64 precision)
    - int64    →  int8 / int16 / int32  (smallest type that fits value range)
    - Other dtypes (category, object, bool) are left unchanged.

    Parameters
    ----------
    df      : DataFrame to downcast (modified in place, copy returned).
    verbose : Print before/after memory usage summary.

    Returns
    -------
    Downcasted copy of df.
    """
    df = df.copy()
    start_mem = df.memory_usage(deep=True).sum() / 1024 ** 2

    for col in df.columns:
        col_dtype = df[col].dtype

        # ── floats ────────────────────────────────────────────────────────
        if col_dtype == np.float64:
            df[col] = df[col].astype(np.float32)

        # ── integers ──────────────────────────────────────────────────────
        elif col_dtype in (np.int64, np.int32):
            col_min = df[col].min()
            col_max = df[col].max()

            if col_min >= np.iinfo(np.int8).min and col_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif col_min >= np.iinfo(np.int16).min and col_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            else:
                df[col] = df[col].astype(np.int32)

        # ── everything else (category, object, bool) — leave as-is ───────

    end_mem = df.memory_usage(deep=True).sum() / 1024 ** 2

    if verbose:
        saved_pct = 100 * (start_mem - end_mem) / start_mem if start_mem else 0
        print(
            f"[Memory] {start_mem:.2f} MB  →  {end_mem:.2f} MB  "
            f"({saved_pct:.1f}% reduction)"
        )

    return df


# ───────────────────────────────────────────────────────────────────────────
# HELPERS — basic lag imputation (unchanged from v1)
# ───────────────────────────────────────────────────────────────────────────

def impute_lag(series: pd.Series) -> pd.Series:
    """
    Fill NaNs with the expanding mean of available values up to that row.
    Used for global (non-region) lag columns.
    """
    expanding_mean = series.expanding(min_periods=1).mean()
    return series.fillna(expanding_mean)


# ───────────────────────────────────────────────────────────────────────────
# [3] SMART IMPUTATION — context-aware grouped lag fallback
# ───────────────────────────────────────────────────────────────────────────

def impute_lag_grouped(
    df: pd.DataFrame,
    col: str,
    group_col: str,
) -> pd.Series:
    """
    Three-tier imputation for region lag columns:

      Tier 1 (preferred):  real shifted value — no fill needed.
      Tier 2 (smart):      historical mean for this (region, day_of_week, hour).
                           Works once any prior week is available.
      Tier 3 (fallback):   region-level expanding mean — always available from
                           the very first row.

    Parameters
    ----------
    df        : Full DataFrame containing 'col', 'group_col', 'day_of_week', 'hour'.
    col       : Name of the lag column already present in df (contains NaNs).
    group_col : Grouping column, typically 'region'.

    Returns
    -------
    pd.Series with zero NaNs.
    """

    result = df[col].copy()

    # ── Tier 2: (region, day_of_week, hour) historical mean ───────────────
    # Compute once across the *entire* df (no leakage risk: lags already use
    # shifted values; the mean here acts like a population prior for the
    # sparse early-window period, not a within-fold average).
    context_mean = (
        df.groupby([group_col, "day_of_week", "hour"])[col]
        .transform("mean")
    )

    # Fill with context mean where lag is NaN
    result = result.fillna(context_mean)

    # ── Tier 3: region expanding mean for anything still NaN ──────────────
    def _region_expanding_mean(grp: pd.Series) -> pd.Series:
        exp = grp.expanding(min_periods=1).mean()
        return grp.fillna(exp)

    still_nan_mask = result.isna()
    if still_nan_mask.any():
        expanding_fill = df.groupby(group_col, sort=False)[col].transform(
            _region_expanding_mean
        )
        result = result.fillna(expanding_fill)

    return result


# ───────────────────────────────────────────────────────────────────────────
# [5] WEATHER DATA — dynamic dates + tenacity retries
# ───────────────────────────────────────────────────────────────────────────

@retry(
    wait=wait_exponential(multiplier=1, min=2, max=10),
    stop=stop_after_attempt(5),
    reraise=True,
)
def _fetch_weather_with_retry(url: str, params: dict) -> dict:
    """
    Inner HTTP call wrapped with tenacity.
    Retries up to 5 times with exponential back-off (2 s → 10 s cap)
    on any requests exception (timeout, connection error, 429, etc.).
    """
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()          # raises HTTPError for 4xx / 5xx
    return resp.json()


def fetch_nyc_weather(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fetch hourly NYC weather from Open-Meteo for the exact date range
    spanned by df.index.  No API key required.

    Changes from v1
    ---------------
    * start_date / end_date are derived dynamically from df.index so the
      function works for any split (train, test, production).
    * The HTTP call is wrapped with @retry (tenacity) to survive transient
      network failures and API rate limits without crashing the pipeline.

    Parameters
    ----------
    df : Any DataFrame whose DatetimeIndex defines the required date range.

    Returns
    -------
    Hourly weather DataFrame indexed by 'time' (America/New_York).
    """
    # [5] Derive dates dynamically from the caller's DataFrame
    start_date = df.index.min().strftime("%Y-%m-%d")
    end_date   = df.index.max().strftime("%Y-%m-%d")

    print(f"  Fetching weather  {start_date}  →  {end_date} ...")

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   40.7128,
        "longitude": -74.0060,
        "start_date": start_date,
        "end_date":   end_date,
        "hourly": ",".join([
            "temperature_2m",
            "apparent_temperature",
            "precipitation",
            "snowfall",
            "snow_depth",
            "windspeed_10m",
            "weathercode",
            "cloudcover",
            "relativehumidity_2m",
        ]),
        "timezone": "America/New_York",
    }

    # [5] Retrying HTTP call
    try:
        data = _fetch_weather_with_retry(url, params)
    except RetryError as exc:
        raise RuntimeError(
            f"Weather API unreachable after 5 retries ({start_date} → {end_date})"
        ) from exc

    weather = pd.DataFrame(data["hourly"])
    weather["time"] = pd.to_datetime(weather["time"])
    weather = weather.set_index("time")

    # ── WMO code → human-readable condition ───────────────────────────────
    def wmo_to_condition(code):
        if pd.isna(code):             return "unknown"
        code = int(code)
        if code in [0, 1]:            return "clear"
        elif code in [2, 3]:          return "cloudy"
        elif code in range(51, 68):   return "rain"
        elif code in range(71, 78):   return "snow"
        elif code in range(80, 83):   return "rain"
        elif code in range(85, 87):   return "snow"
        elif code in [95, 96, 99]:    return "storm"
        else:                         return "other"

    weather["weather_condition"] = weather["weathercode"].apply(wmo_to_condition)
    weather["is_raining"]        = (weather["precipitation"] > 0).astype(int)
    weather["is_snowing"]        = (weather["snowfall"]      > 0).astype(int)
    weather["is_heavy_weather"]  = (
        (weather["precipitation"] > 5) | (weather["snowfall"] > 2)
    ).astype(int)
    weather["is_freezing"]       = (weather["temperature_2m"] < 0).astype(int)
    weather["temp_bin"]          = pd.cut(
        weather["temperature_2m"],
        bins=[-40, 0, 10, 40],
        labels=["cold", "mild", "warm"],
    )
    # Winter Storm Jonas — Jan 22-25, NYC record snowfall ~26.8 in
    weather["is_storm_jonas"] = (
        (weather.index >= "2016-01-22") &
        (weather.index <= "2016-01-25")
    ).astype(int)

    return weather


# ───────────────────────────────────────────────────────────────────────────
# HOLIDAYS & EVENTS — NYC Jan–Mar 2016
# ───────────────────────────────────────────────────────────────────────────

NYC_HOLIDAYS_2016 = {
    pd.Timestamp("2016-01-01"): "new_years_day",
    pd.Timestamp("2016-01-18"): "mlk_day",
    pd.Timestamp("2016-02-15"): "presidents_day",
    pd.Timestamp("2016-02-14"): "valentines_day",
    pd.Timestamp("2016-03-17"): "st_patricks_day",
    pd.Timestamp("2016-03-25"): "good_friday",
    pd.Timestamp("2016-03-27"): "easter_sunday",
}

NYC_EVENTS_2016 = {
    "restaurant_week":   ("2016-01-18", "2016-01-29"),
    "fashion_week":      ("2016-02-11", "2016-02-18"),
    "nyc_half_marathon": ("2016-03-20", "2016-03-20"),
}


def build_holiday_features(df: pd.DataFrame) -> pd.DataFrame:
    date_only = df.index.normalize()
    df["is_holiday"]   = date_only.isin(NYC_HOLIDAYS_2016.keys()).astype(int)
    df["holiday_name"] = date_only.map(NYC_HOLIDAYS_2016).fillna("none")

    holiday_dates = sorted(NYC_HOLIDAYS_2016.keys())

    def days_to_next(ts):
        future = [h for h in holiday_dates if h >= ts.normalize()]
        return (future[0] - ts.normalize()).days if future else 99

    def days_after_prev(ts):
        past = [h for h in holiday_dates if h <= ts.normalize()]
        return (ts.normalize() - past[-1]).days if past else 99

    df["days_to_holiday"]    = [days_to_next(t)    for t in df.index]
    df["days_after_holiday"] = [days_after_prev(t) for t in df.index]

    for event_name, (start, end) in NYC_EVENTS_2016.items():
        df[f"is_{event_name}"] = (
            (df.index >= pd.Timestamp(start)) &
            (df.index <= pd.Timestamp(end) + pd.Timedelta(hours=23))
        ).astype(int)

    return df


# ───────────────────────────────────────────────────────────────────────────
# CORE FEATURE BUILDER — zero dropna(), smart imputation applied
# ───────────────────────────────────────────────────────────────────────────

def build_features(
    raw_df:     pd.DataFrame,
    weather_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Full feature engineering pipeline.
    Zero rows are dropped — all NaNs are imputed via the three-tier strategy.

    Parameters
    ----------
    raw_df     : pickup DataFrame, DatetimeIndex, 'total_pickups' column.
    weather_df : output of fetch_nyc_weather().
    """
    df = raw_df.copy()
    df.index = pd.to_datetime(df.index)

    # ── 1. Basic time features ─────────────────────────────────────────────
    df["hour"]        = df.index.hour
    df["day"]         = df.index.day
    df["month"]       = df.index.month
    df["day_of_week"] = df.index.dayofweek     # 0=Mon … 6=Sun
    df["is_weekend"]  = (df.index.dayofweek >= 5).astype(int)

    # ── 2. Cyclical encoding ───────────────────────────────────────────────
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"]        / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"]        / 24)
    df["dow_sin"]   = np.sin(2 * np.pi * df["day_of_week"] /  7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["day_of_week"] /  7)
    df["day_sin"]   = np.sin(2 * np.pi * df["day"]         / 31)
    df["day_cos"]   = np.cos(2 * np.pi * df["day"]         / 31)
    df["month_sin"] = np.sin(2 * np.pi * df["month"]       / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"]       / 12)

    # ── 3. Peak / night flags ──────────────────────────────────────────────
    df["is_peak_hour"] = df["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df["is_night"]     = df["hour"].isin([0, 1, 2, 3, 4, 5]).astype(int)

    # ── 4. Global lag features (NaNs filled, availability flags added) ─────
    lag_shifts = {
        "lag_1":   1,
        "lag_24":  24,
        "lag_168": 168,
        "lag_336": 336,
        "lag_720": 720,
    }

    for col, shift in lag_shifts.items():
        raw_lag        = df["total_pickups"].shift(shift)
        available_flag = raw_lag.notna().astype(int)
        df[col]        = impute_lag(raw_lag)
        if shift >= 24:
            df[f"{col}_available"] = available_flag

    # ── 5. Rolling features (min_periods=1, no NaN) ────────────────────────
    base = df["total_pickups"].shift(1)      # prevent leakage

    df["rolling_mean_24"]  = base.rolling(24,  min_periods=1).mean()
    df["rolling_std_24"]   = base.rolling(24,  min_periods=1).std().fillna(0)
    df["rolling_mean_168"] = base.rolling(168, min_periods=1).mean()
    df["rolling_std_168"]  = base.rolling(168, min_periods=1).std().fillna(0)
    df["rolling_mean_336"] = base.rolling(336, min_periods=1).mean()
    df["rolling_mean_720"] = base.rolling(720, min_periods=1).mean()

    # ── 6. Weekend / weekday conditioned rolling ───────────────────────────
    weekend_mask = df.index.dayofweek >= 5
    weekday_mask = ~weekend_mask

    df["rolling_mean_weekend_24"] = (
        base.where(weekend_mask)
        .rolling(24, min_periods=1)
        .mean()
        .fillna(df["rolling_mean_24"])
    )
    df["rolling_mean_weekday_24"] = (
        base.where(weekday_mask)
        .rolling(24, min_periods=1)
        .mean()
        .fillna(df["rolling_mean_24"])
    )

    # ── 7. Holiday features ────────────────────────────────────────────────
    df = build_holiday_features(df)

    holiday_base = base.where(df["is_holiday"].astype(bool))
    df["rolling_mean_holiday_24"] = (
        holiday_base
        .rolling(24, min_periods=1)
        .mean()
        .fillna(df["rolling_mean_24"])
    )

    # ── 8. Region lag & rolling — [3] smart imputation applied ────────────
    if "region" in df.columns:

        df = df.sort_index()          # monotone time order within each region

        grp_col = "region"

        for col, shift in [
            ("region_lag_1",   1),
            ("region_lag_24",  24),
            ("region_lag_168", 168),
            ("region_lag_336", 336),
        ]:
            raw_lag = df.groupby(grp_col, sort=False)["total_pickups"].shift(shift)
            df[col] = impute_lag_grouped(
                df.assign(**{col: raw_lag}), col, grp_col
            )
            if shift >= 24:
                df[f"{col}_available"] = raw_lag.notna().astype(int)

        # Region rolling — min_periods=1, no NaN
        df["_base_r"] = (
            df.groupby(grp_col, sort=False)["total_pickups"].shift(1)
        )

        df["region_rolling_mean_24"] = (
            df.groupby(grp_col, sort=False)["_base_r"]
            .transform(lambda x: x.rolling(24, min_periods=1).mean())
        )
        df["region_rolling_std_24"] = (
            df.groupby(grp_col, sort=False)["_base_r"]
            .transform(lambda x: x.rolling(24, min_periods=1).std().fillna(0))
        )
        df["region_rolling_mean_168"] = (
            df.groupby(grp_col, sort=False)["_base_r"]
            .transform(lambda x: x.rolling(168, min_periods=1).mean())
        )

        df = df.drop(columns=["_base_r"])
        df = df.sort_index()          # restore global time order

    # ── 9. Weather features ────────────────────────────────────────────────
    weather_cols = [
        "temperature_2m", "apparent_temperature",
        "precipitation", "snowfall", "snow_depth",
        "windspeed_10m", "cloudcover", "relativehumidity_2m",
        "is_raining", "is_snowing", "is_heavy_weather",
        "is_freezing", "is_storm_jonas",
        "temp_bin", "weather_condition",
    ]
    df = df.join(weather_df[weather_cols], how="left")

    df[weather_cols] = df[weather_cols].ffill(limit=2)

    for col in weather_cols:
        if df[col].isna().any():
            if df[col].dtype.kind in "ifc":
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna(df[col].mode().iloc[0])

    # ── 10. Interaction features ───────────────────────────────────────────
    df["weekend_hour_interaction"] = df["hour"]           * df["is_weekend"]
    df["rain_x_peak_hour"]         = df["is_raining"]     * df["is_peak_hour"]
    df["snow_x_night"]             = df["is_snowing"]     * df["is_night"]
    df["holiday_x_hour"]           = df["is_holiday"]     * df["hour"]
    df["temp_x_weekend"]           = df["temperature_2m"] * df["is_weekend"]
    df["storm_x_peak"]             = df["is_storm_jonas"] * df["is_peak_hour"]
    df["wind_x_rain"]              = df["windspeed_10m"]  * df["is_raining"]

    # ── 11. Sanity check ───────────────────────────────────────────────────
    nan_counts = df.isnull().sum()
    remaining  = nan_counts[nan_counts > 0]
    if not remaining.empty:
        print("[WARNING] NaN still present after imputation:")
        print(remaining)
    else:
        print(f"[OK] Zero NaN rows. Shape: {df.shape}")

    return df


# ───────────────────────────────────────────────────────────────────────────
# WAPE METRIC
# ───────────────────────────────────────────────────────────────────────────

def wape(y_true, y_pred):
    return (
        np.sum(np.abs(y_true - y_pred)) /
        np.sum(np.abs(y_true))
    ) * 100


# ───────────────────────────────────────────────────────────────────────────
# PIPELINE ENTRY POINT
# ───────────────────────────────────────────────────────────────────────────

# ── Fetch weather (dates derived automatically from each split) ────────────
print("Fetching NYC weather for train split ...")
weather_df = fetch_nyc_weather(train_df)     # [5] dynamic date extraction
print(f"  Weather shape: {weather_df.shape}")

# ── Build features ─────────────────────────────────────────────────────────
print("\nBuilding train features ...")
df = build_features(train_df, weather_df)

print("\nBuilding test features ...")
# [5] Test weather fetched for its own date range, not hard-coded
test_weather_df = fetch_nyc_weather(test_df)
test_processed  = build_features(test_df, test_weather_df)

# ── [4] Downcast memory before splitting into X / y ───────────────────────
print("\nReducing memory usage ...")
df             = reduce_mem_usage(df)
test_processed = reduce_mem_usage(test_processed)

# ── Features / target ─────────────────────────────────────────────────────
X_full = df.drop(columns=["total_pickups"])
y_full = df["total_pickups"]

X_test = test_processed.drop(columns=["total_pickups"])
y_test = test_processed["total_pickups"]

# ── Categorical columns ────────────────────────────────────────────────────
categorical_cols = ["region", "holiday_name", "weather_condition", "temp_bin"]

for col in categorical_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].astype("category")
    if col in X_test.columns:
        X_test[col] = X_test[col].astype("category")

# ───────────────────────────────────────────────────────────────────────────
# [1] TIME-SERIES CV WITH GAP
# ─────────────────────────────────────────────────────────────────────────
# gap=8 skips 8 intervals (8 × 15 min = 2 hours) between the end of each
# training fold and the start of the validation fold, simulating the real
# pipeline latency that exists between data collection and model scoring.
# Without the gap, the last training rows (which share temporal context with
# the first validation rows) would cause leakage and inflate CV metrics.
# ──────────────────────────────────────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=5, gap=8)   # [1] edge-leakage prevention

print(f"\nDataset has {len(X_full)} rows for CV.")
print(f"Min fold size at n_splits=5, gap=8: ~{len(X_full) // 6} rows")

# ───────────────────────────────────────────────────────────────────────────
# MLFLOW EXPERIMENT
# ───────────────────────────────────────────────────────────────────────────

mlflow.set_experiment("lightgbm_full_features_v2")

# ───────────────────────────────────────────────────────────────────────────
# [2] OPTUNA OBJECTIVE — with LightGBMPruningCallback
# ───────────────────────────────────────────────────────────────────────────

def objective(trial):
    """
    Optuna objective.

    [2] LightGBMPruningCallback reports the per-fold validation RMSE after
    every boosting round to the Optuna pruner.  Unpromising trials are
    stopped mid-training (not just after a full fold), cutting wall-clock
    time significantly when n_trials is large.
    """
    rmse_scores = []

    params = {
        "n_estimators":      trial.suggest_int("n_estimators",       200, 1000, step=50),
        "learning_rate":     trial.suggest_float("learning_rate",    1e-2, 0.1, log=True),
        "max_depth":         trial.suggest_int("max_depth",          3, 7),
        "num_leaves":        trial.suggest_int("num_leaves",         20, 80, step=5),
        "min_child_samples": trial.suggest_int("min_child_samples",  10, 40, step=5),
        "subsample":         trial.suggest_float("subsample",        0.7, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha",        1e-4, 5, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda",       1e-4, 5, log=True),
        "objective":         "regression",
        "random_state":      42,
        "n_jobs":            -1,
        "verbosity":         -1,
    }

    active_cat = [c for c in categorical_cols if c in X_full.columns]

    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
        X_tr, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="rmse",
            categorical_feature=active_cat,
            callbacks=[
                lgb.early_stopping(50),
                lgb.log_evaluation(0),
                # [2] Pruning: report intermediate RMSE to Optuna each round.
                # The `period` arg controls reporting frequency (default=1).
                LightGBMPruningCallback(trial, "rmse"),
            ],
        )

        y_pred = model.predict(X_val)
        rmse_scores.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmse_scores)


# ───────────────────────────────────────────────────────────────────────────
# OPTUNA STUDY
# ─────────────────────────────────────────────────────────────────────────
# MedianPruner: prune if the trial's intermediate value is worse than the
# median of completed trials at the same step.  Pairs naturally with
# LightGBMPruningCallback above.
# ──────────────────────────────────────────────────────────────────────────
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=50)
study  = optuna.create_study(direction="minimize", pruner=pruner)
study.optimize(objective, n_trials=50)

print("\nBest params:")
print(study.best_params)
print(f"\nBest CV RMSE: {study.best_value:.4f}")

pruned   = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f"Trials — complete: {complete}  pruned: {pruned}")

# ───────────────────────────────────────────────────────────────────────────
# FINAL MODEL
# ───────────────────────────────────────────────────────────────────────────

active_cat  = [c for c in categorical_cols if c in X_full.columns]

final_model = LGBMRegressor(
    **study.best_params,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
final_model.fit(X_full, y_full, categorical_feature=active_cat)

# ───────────────────────────────────────────────────────────────────────────
# EVALUATION
# ───────────────────────────────────────────────────────────────────────────

y_test_pred = final_model.predict(X_test)

final_mae  = mean_absolute_error(y_test, y_test_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
final_wape = wape(y_test, y_test_pred)
final_r2   = r2_score(y_test, y_test_pred)

print("\n--- FINAL TEST RESULTS ---")
print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"WAPE : {final_wape:.4f}%")
print(f"R2   : {final_r2:.4f}")

# ───────────────────────────────────────────────────────────────────────────
# FEATURE IMPORTANCES
# ───────────────────────────────────────────────────────────────────────────

feat_imp = pd.Series(
    final_model.feature_importances_,
    index=X_full.columns,
).sort_values(ascending=False)

print("\n--- TOP 20 FEATURE IMPORTANCES ---")
print(feat_imp.head(20).to_string())

# ───────────────────────────────────────────────────────────────────────────
# MLFLOW LOGGING
# ───────────────────────────────────────────────────────────────────────────

with mlflow.start_run(run_name="lgbm_full_features_v2"):

    mlflow.log_params(study.best_params)
    mlflow.log_metric("FINAL_MAE",          final_mae)
    mlflow.log_metric("FINAL_RMSE",         final_rmse)
    mlflow.log_metric("FINAL_WAPE",         final_wape)
    mlflow.log_metric("FINAL_R2",           final_r2)
    mlflow.log_param("n_features",          X_full.shape[1])
    mlflow.log_param("n_train_rows",        len(X_full))
    mlflow.log_param("cv_gap_intervals",    8)
    mlflow.log_param("optuna_pruned_trials", pruned)

    mlflow.lightgbm.log_model(final_model, artifact_path="model")

print("\nModel logged to MLflow successfully.")

Fetching NYC weather for train split ...
  Fetching weather  2016-01-01  →  2016-02-29 ...
  Weather shape: (1440, 16)

Building train features ...
[WARNING] NaN still present after imputation:
lag_1                        1
lag_24                      24
lag_168                    168
lag_336                    336
lag_720                    720
rolling_mean_24              1
rolling_mean_168             1
rolling_mean_336             1
rolling_mean_720             1
rolling_mean_weekend_24      1
rolling_mean_weekday_24      1
rolling_mean_holiday_24      1
region_rolling_mean_24      30
region_rolling_mean_168     30
dtype: int64

Building test features ...
  Fetching weather  2016-03-01  →  2016-03-31 ...
[WARNING] NaN still present after imputation:
lag_1                        1
lag_24                      24
lag_168                    168
lag_336                    336
lag_720                    720
rolling_mean_24              1
rolling_mean_168             1
rolling_mean_336  

2026/05/28 17:23:04 INFO mlflow.tracking.fluent: Experiment with name 'lightgbm_full_features_v2' does not exist. Creating a new experiment.
[I 2026-05-28 17:23:04,068] A new study created in memory with name: no-name-3ddd9830-72d2-4ea4-89f3-7862cee814e5



Dataset has 172680 rows for CV.
Min fold size at n_splits=5, gap=8: ~28780 rows
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[300]	valid_0's rmse: 21.4953	valid_0's l2: 462.048


c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 0 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 2 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 3 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 4 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: Use

Training until validation scores don't improve for 50 rounds


c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 74 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 75 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 76 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 77 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509: UserWarning: The reported value is ignored because this `step` 78 is already reported.
  optuna_warn(
c:\Users\Soham\anaconda3\envs\myenv\lib\site-packages\optuna\trial\_trial.py:509

... (364 outputs left out)

Training until validation scores don't improve for 50 rounds


[I 2026-05-28 17:31:26,576] Trial 49 pruned. Trial was pruned at iteration 50.



Best params:
{'n_estimators': 800, 'learning_rate': 0.05444123476418912, 'max_depth': 7, 'num_leaves': 55, 'min_child_samples': 20, 'subsample': 0.7283622814542228, 'colsample_bytree': 0.7581367586919908, 'reg_alpha': 0.006267028144374844, 'reg_lambda': 0.0001425134442113599}

Best CV RMSE: 19.0333
Trials — complete: 17  pruned: 33

--- FINAL TEST RESULTS ---
MAE  : 12.4123
RMSE : 20.0499
WAPE : 9.4185%
R2   : 0.9764

--- TOP 20 FEATURE IMPORTANCES ---
region                   3158
lag_1                    2940
lag_2                    2303
lag_4                    2061
lag_3                    1957
hour                     1574
region_lag_1             1334
avg_pickups              1298
lag_24                   1268
lag_168                  1219
lag_720                  1190
hour_sin                 1170
hour_cos                 1077
rolling_std_24           1035
lag_336                   990
rolling_std_168           911
region_lag_336            857
region_lag_168            783
re

Exception: Run with UUID 9d062824b3cc419f867cb4d469b2e41e is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

In [19]:
# ==========================================================
# SAVE DEPLOYMENT PIPELINE
# ==========================================================

import joblib

deployment_artifact = {

    "model": final_model,

    "feature_columns": (
        X_full.columns.tolist()
    ),

    "categorical_columns": (
        active_cat
    ),

    "best_params": (
        study.best_params
    )

}

joblib.dump(

    deployment_artifact,

    "lgbm_taxi_pipeline.pkl"
)

print(
    "\nDeployment pipeline saved successfully."
)


Deployment pipeline saved successfully.


In [6]:
# ==========================================================
# CREATE HISTORICAL SEED DATA
# ==========================================================

# number of unique regions
NUM_REGIONS = df["region"].nunique()

# last 30 days of 15-minute data
seed_rows = 30 * 24 * 4 * NUM_REGIONS

historical_seed_data = (
    df
    .sort_index()
    .tail(seed_rows)
)

# save
historical_seed_data.to_csv(
    "historical_seed_data.csv",
    index=True
)

print(
    "\nHistorical seed data saved successfully."
)

print(
    f"Shape: {historical_seed_data.shape}"
)


Historical seed data saved successfully.
Shape: (86400, 78)
